In [4]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
import joblib
import pandas as pd


In [8]:
import pandas as pd

# 1) LOAD TRANSACTIONS + LABELS
transactions = pd.read_csv("../data/transactions_data.csv")
fraud = pd.read_json("../data/train_fraud_labels.json")

# map labels Yes/No -> 1/0  (adjust if your json is already 0/1)
fraud['target'] = fraud['target'].map({'Yes': 1, 'No': 0})

# labels are index-aligned with transactions rows, so we reindex
transactions['target'] = (
    fraud['target']
    .reindex(transactions.index)
    .fillna(0)
    .astype(int)
)

# 2) BASIC CLEANING + FEATURE ENGINEERING (light version)
transactions['amount'] = (
    transactions['amount']
    .replace(r'[$,]', '', regex=True)
    .astype(float)
)

transactions['date'] = pd.to_datetime(transactions['date'], errors='coerce')
transactions['hour'] = transactions['date'].dt.hour
transactions['dayofweek'] = transactions['date'].dt.dayofweek
transactions['month'] = transactions['date'].dt.month
transactions['use_chip'] = transactions['use_chip'].apply(lambda x: 1 if 'Chip' in str(x) else 0)

data_full = transactions

# 3) DEFINE feature_cols BEFORE USING IT  🔴 THIS WAS MISSING
feature_cols = [
    "amount",
    "use_chip",
    "merchant_state",
    "mcc",
    "errors",
    "hour",
    "dayofweek",
    "month",
]

# 4) BALANCE THE DATASET ON THIS data_full
fraud_rows = data_full[data_full["target"] == 1]
nonfraud_rows = data_full[data_full["target"] == 0].sample(
    n=min(len(data_full), 10 * len(fraud_rows)),  # 10x nonfraud
    random_state=42,
)

data_balanced = pd.concat([fraud_rows, nonfraud_rows]).sample(
    frac=1, random_state=42
).reset_index(drop=True)

X_bal = data_balanced[feature_cols]
y_bal = data_balanced["target"]

print("Balanced shape:", X_bal.shape)
print("Balanced target counts:\n", y_bal.value_counts())


Balanced shape: (43879, 8)
Balanced target counts:
 target
0    39890
1     3989
Name: count, dtype: int64


In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_bal, y_bal,
    test_size=0.2,
    stratify=y_bal,
    random_state=42
)

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

num_cols = X_train.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
    ]
)

rf_model = RandomForestClassifier(
    n_estimators=200,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42,
)

rf_clf = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", rf_model),
])

print("🌲 Training Random Forest...")
rf_clf.fit(X_train, y_train)


🌲 Training Random Forest...


,steps,"[('preprocess', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [11]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

y_proba_rf = rf_clf.predict_proba(X_test)[:, 1]
y_pred_rf = (y_proba_rf > 0.5).astype(int)

print(classification_report(y_test, y_pred_rf))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_rf))


              precision    recall  f1-score   support

           0       0.94      0.77      0.84      7978
           1       0.18      0.52      0.27       798

    accuracy                           0.74      8776
   macro avg       0.56      0.64      0.56      8776
weighted avg       0.87      0.74      0.79      8776

Confusion Matrix:
 [[6107 1871]
 [ 381  417]]
ROC-AUC: 0.7107723715154017


In [14]:
import joblib

joblib.dump(rf_clf, "../models/fraud_detection_rf_pipeline.pkl")
print("✅ Saved Random Forest pipeline to models/fraud_detection_rf_pipeline.pkl")


✅ Saved Random Forest pipeline to models/fraud_detection_rf_pipeline.pkl
